In [0]:
import pandas as pd

TASK 1: Load & Inspect the Dataset

In [0]:
#1 Load the CSV with the correct sep and decimal arguments.
df = spark.read.table("brightlearn.default.bright_coffee_shop").toPandas()

df.head()

In [0]:
#2 Print the shape, all column names, and data types.

print(df.shape)

df.sample(10)

In [0]:
#2 all column names and data types
df.info()

In [0]:
#3 Check for missing values - no missing values
df.isnull().sum()

In [0]:
#4 
df.describe()

In [0]:
#4
df.describe(include='object')

In [0]:
#5 Print value counts
df["product_category"].value_counts()

In [0]:
#5 Print value counts
df["store_location"].value_counts()

TASK 2: Feature Engineering & Distributions

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px # the easy, high-level interface
import plotly.graph_objects as go # the advanced, full-control interface

In [0]:
#1. Create a revenue column: unit_price * transaction_qty
df["Revenue"] = df["transaction_qty"]*df["unit_price"]
df.head()

In [0]:
#2. Extract hour from transaction_time and month from transaction_date

# Extract hour from transaction_time
df['hour'] = df['transaction_time'].dt.hour

# Extract month from transaction_date
df['month'] = pd.to_datetime(df['transaction_date']).dt.month

df.head()


In [0]:
#3. Plot the distribution of unit_price (histogram with KDE)

sns.histplot(df, x='unit_price', kde=True, color='steelblue', bins=30)
plt.title('Distribution of Unit Price')
plt.xlabel('Unit Price (USD)')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

#Describe the shape
##strongly right-skewed (positively skewed)

In [0]:
#4. Plot total revenue by store_location as a bar chart.

# Group by store location and sum revenue
store_rev = df.groupby('store_location')['Revenue'].sum().sort_values(ascending=False)

# Create bar chart
plt.figure(figsize=(10, 6))
plt.bar(store_rev.index, store_rev.values, color='teal', edgecolor='white')
plt.title('Total Revenue by Store Location', fontsize=14)
plt.xlabel('Store Location')
plt.ylabel('Revenue (USD)')
plt.xticks(rotation=45, ha='right')

# Add value labels on top of bars
for i, v in enumerate(store_rev.values):
    plt.text(i, v + 3000, f'${v:,.0f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()


In [0]:
#5. Plot transactions by hour of day. When is the shop busiest?

# Count transactions by hour
hourly_transactions = df.groupby('hour').size().sort_index()

# Create bar chart
plt.figure(figsize=(12, 6))
plt.bar(hourly_transactions.index, hourly_transactions.values, color='coral', edgecolor='white')
plt.title('Transactions by Hour of Day', fontsize=14)
plt.xlabel('Hour of Day')
plt.ylabel('Number of Transactions')
plt.xticks(range(0, 24))
plt.grid(axis='y', alpha=0.3)

# Add value labels on top of bars
for i, v in enumerate(hourly_transactions.values):
    plt.text(hourly_transactions.index[i], v + 100, str(v), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

# busiest hour
busiest_hour = hourly_transactions.idxmax()
max_transactions = hourly_transactions.max()
print(f"\nBusiest hour: {busiest_hour}:00 with {max_transactions:,} transactions")


TASK 03 Correlation & Group Analysis

In [0]:
#1. Create a seaborn correlation heatmap for unit_price, transaction_qty, and revenue.

sns.heatmap(df[["unit_price","transaction_qty","Revenue"]].corr(numeric_only=True, method = "spearman"), annot=True, fmt=".2f", cmap="coolwarm")

In [0]:
#2. Build a pivot table: total revenue by store_location (rows) × product_category (columns).

pd.pivot_table(df, 
               values="Revenue",
               index="store_location",
               columns="product_category",
               aggfunc="sum")

In [0]:
#3. Visualise the pivot table as a grouped or stacked bar chart.

pivot = pd.pivot_table(df, 
                       values="Revenue",
                       index="store_location",
                       columns="product_category",
                       aggfunc="sum")

# Create figure with adjusted size
fig, ax = plt.subplots(figsize=(12, 6))
pivot.plot(kind='bar', stacked=True, ax=ax)

# Move legend to the right side, outside the plot
ax.legend(title='Product Category', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.set_xlabel('Store Location')
ax.set_ylabel('Revenue (USD)')
ax.set_title('Revenue by Store Location and Product Category')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show() 

In [0]:
#4. Which store earns the most? Which product category drives the most revenue overall?

# Revenue by store location
store_revenue = df.groupby('store_location')['Revenue'].sum().sort_values(ascending=False)
print("Revenue by Store Location:")
print(store_revenue)
print(f"\nTop earning store: {store_revenue.index[0]} with ${store_revenue.iloc[0]:,.2f}")

print("\n" + "="*60 + "\n")

# Revenue by product category
category_revenue = df.groupby('product_category')['Revenue'].sum().sort_values(ascending=False)
print("Revenue by Product Category:")
print(category_revenue)
print(f"\nTop revenue category: {category_revenue.index[0]} with ${category_revenue.iloc[0]:,.2f}")

Task 04 Visualisation Library Comparison

In [0]:
#Reproduce the revenue by product category bar chart in all three libraries:
• matplotlib
• seaborn
• plotly

In [0]:
cat_rev = df.groupby('product_category')['Revenue'].sum().sort_values(ascending=False)
cats = cat_rev.index
vals = cat_rev.values

In [0]:
# 1. MATPLOTLIB
import matplotlib.pyplot as plt
cat_rev = df.groupby('product_category')['Revenue'].sum().reset_index()
cats = cat_rev['product_category']
vals = cat_rev['Revenue']
plt.figure(figsize=(10,5))
plt.bar(cats, vals, color='steelblue', edgecolor='white')
plt.title('Revenue by Product Category')
plt.xlabel('Category'); plt.ylabel('Revenue (USD)')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

In [0]:
# 2. SEABORN
import seaborn as sns
sns.set_theme(style='darkgrid')
ax = sns.barplot(x=cats, y=vals, palette='Blues_d')
ax.set_title('Revenue by Product Category')
ax.set_xlabel('Category'); ax.set_ylabel('Revenue (USD)')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

In [0]:
# 3. PLOTLY
import plotly.express as px
cat_df = cat_rev.copy() # px needs a DataFrame
cat_df.columns = ['Category','Revenue'] # rename for clarity
fig = px.bar(cat_df, x='Category', y='Revenue',
title='Revenue by Product Category')
fig.update_layout(xaxis_title='Category', yaxis_title='Revenue (USD)')
fig.show()